## Grouping check: does SCC do better when look-alike grades are merged?

Nothing is retrained here. During the CNN retrain, each model saved its predictions on the test cells (3 models x 3 seeds = 9 `probs_*.npz` files in the "Re trained models" dataset). This cell reuses those saved predictions.

For every test cell, the six scores (ASC-H, ASC-US, HSIL, LSIL, Negative, SCC) are added into groups, and the biggest group wins:

- **Three tiers:** Negative | low (ASC-US + LSIL) | high (ASC-H + HSIL + SCC)
- **Four classes:** Negative | ASC-US + LSIL | ASC-H + HSIL | SCC on its own

What it prints, per file and averaged by model:

- SCC recall in the six-class view (called exactly SCC)
- share of SCC cells that land in the high tier, and share called Negative
- SCC recall in the four-class view
- recall of the high tier and the low tier, and how often each is called Negative
- accuracy and macro F1, six-class and three-tier

Merging classes makes the scores look better by construction, so the six-class result stays the main result. The three-tier numbers are a secondary view. They answer a different question, so they should not be read as the model improving.

Needs the "Re trained models" dataset added as an input.

In [3]:
# Grouping check: SCC and tier recall from the SAVED probabilities. Nothing is retrained.
# Needs the Kaggle Dataset with the nine probs_*.npz files attached ("Re trained models").
import glob, os, re
import numpy as np
from sklearn.metrics import f1_score

CLASSES = ["ASC-H", "ASC-US", "HSIL", "LSIL", "Negative", "SCC"]   # order of the probability columns
SCC, NEG = 5, 4
TIER3 = {"Negative": [4], "low (ASC-US+LSIL)": [1, 3], "high (ASC-H+HSIL+SCC)": [0, 2, 5]}
GROUP4 = {"Negative": [4], "ASC-US+LSIL": [1, 3], "ASC-H+HSIL": [0, 2], "SCC": [5]}

files = sorted(glob.glob("/kaggle/input/**/probs_*.npz", recursive=True))
assert files, "No probs_*.npz found. Add the retrain results Dataset as an input."
print(len(files), "files found")
z0 = np.load(files[0], allow_pickle=True)
print("keys inside one file:", list(z0.keys()))

def pick(z, names):
    for n in names:
        if n in z.files: return z[n]
    raise KeyError(f"none of {names} in {z.files}. Edit the names in pick() to match the keys printed above.")

def merge(p, groups):                      # add the probabilities of the classes in each group
    return np.stack([p[:, idx].sum(axis=1) for idx in groups.values()], axis=1)

def labels(y, groups):
    out = np.zeros(len(y), dtype=int)
    for g, idx in enumerate(groups.values()):
        out[np.isin(y, idx)] = g
    return out

rows = []
for f in files:
    z = np.load(f, allow_pickle=True)
    y = np.asarray(pick(z, ["y_true", "true", "labels", "y", "y_test"])).astype(int)
    p = np.asarray(pick(z, ["probs", "y_prob", "y_pred_probs", "test_probs", "prob", "p"]))
    name = os.path.basename(f).lower()
    model = next((m for k, m in [("eff", "EfficientNetB0"), ("conv", "ConvNeXt"), ("reg", "RegNet")] if k in name), name)
    pred6 = p.argmax(1)
    P3 = merge(p, TIER3).argmax(1); Y3 = labels(y, TIER3)
    P4 = merge(p, GROUP4).argmax(1); Y4 = labels(y, GROUP4)
    scc = y == SCC
    high = np.isin(y, TIER3["high (ASC-H+HSIL+SCC)"])
    rows.append(dict(file=os.path.basename(f), model=model,
        scc_recall_6class=(pred6[scc] == SCC).mean(),
        scc_in_high_tier=(P3[scc] == 2).mean(),
        scc_called_negative=(P3[scc] == 0).mean(),
        scc_recall_4class=(P4[scc] == 3).mean(),
        high_tier_recall=(P3[high] == 2).mean(),
        high_called_negative=(P3[high] == 0).mean(),
        low_tier_recall=(P3[Y3 == 1] == 1).mean(),
        low_called_negative=(P3[Y3 == 1] == 0).mean(),
        neg_recall=(P3[Y3 == 0] == 0).mean(),
        acc_3tier=(P3 == Y3).mean(), macroF1_3tier=f1_score(Y3, P3, average="macro"),
        acc_6class=(pred6 == y).mean(), macroF1_6class=f1_score(y, pred6, average="macro")))

import pandas as pd
df = pd.DataFrame(rows)
pd.set_option("display.width", 220)
print("\nPer file:"); print(df.round(3).to_string(index=False))
print("\nMean over seeds, by model (± sd):")
g = df.drop(columns="file").groupby("model")
print((g.mean().round(3).astype(str) + " ± " + g.std().round(3).astype(str)).T.to_string())
print("\nAll runs together (mean):"); print(df.drop(columns=["file", "model"]).mean().round(3).to_string())
print("\nTest cells per class:", {c: int((y == i).sum()) for i, c in enumerate(CLASSES)})

9 files found
keys inside one file: ['y_prob', 'y_true', 'test_keys']

Per file:
                          file          model  scc_recall_6class  scc_in_high_tier  scc_called_negative  scc_recall_4class  high_tier_recall  high_called_negative  low_tier_recall  low_called_negative  neg_recall  acc_3tier  macroF1_3tier  acc_6class  macroF1_6class
      probs_ConvNeXt_seed1.npz       ConvNeXt              0.441             0.971                0.000              0.412             0.960                 0.018            0.717                0.207       0.926      0.907          0.866       0.833           0.618
      probs_ConvNeXt_seed2.npz       ConvNeXt              0.676             0.912                0.029              0.676             0.972                 0.014            0.709                0.173       0.939      0.917          0.878       0.841           0.636
      probs_ConvNeXt_seed3.npz       ConvNeXt              0.441             0.912                0.000              0